### Fig 2B : Time Traj , GMM Hist for selected networks

In [ ]:
ENV["OMP_NUM_THREADS"] = "1"

using HDF5
using Plots
using Measures
using PyCall
using Statistics
using LinearAlgebra
using StatsBase
using LaTeXStrings

sklearn = pyimport("sklearn.mixture")

gr()

# =======================================================
# File Paths
# =======================================================
traj_file = raw"F:\HR_Project\3nets_fullres\multimodal_search_3nets_fullres_V3.h5"
hist_file = raw"F:\HR_Project\3nets_fullres\multimodal_search_3nets_fullres_V3.h5"
frac_file = raw"F:\HR_Project\3nets_fullres\fraction_data_3nets_fullres_V3.h5"  

# Opposite-IC data 
OPIC_DIR = raw"F:\HR_Project\noise1_op_ic"

# Raw per-trial op-IC data 
opic_hist_file_for_net = Dict(
    "net_31"   => "multimodal_search_net_1_op_ic.h5",
    "net_1762" => "multimodal_search_net_9_op_ic.h5",
)

# Aggregate on_fraction/off_fraction summary files
opic_frac_file_for_net = Dict(
    "net_31"   => "fraction_data_net_1_op_ic.h5",
    "net_1762" => "fraction_data_net_9_op_ic.h5",
)

# =======================================================
# Configs -- 2 HR-enabling examples
# =======================================================
configs = [
    (net="net_1762", param="param_72", ic="ic_1"), # 120
    (net="net_31",   param="param_96", ic="ic_1"), #91,31
]

target_trials = ["trial_1", "trial_2"]
node_idx      = 1

# =======================================================
# Plot style
# =======================================================
TITLE_SIZE  = 30
LABEL_SIZE  = 30
TICK_SIZE   = 25
LEGEND_SIZE = 25
LINE_W_TRAJ = 3.5
ALPHA_UNI   = 0.85
ALPHA_BI    = 0.75
COLOR_GREY  = :gray45
COLOR_BI_1  = "#2166AC"
COLOR_BI_2  = "#D6604D"
TRAJ_XLIMS  = (1000.0, 1800.0)
TRAJ_COLORS = [:hotpink, :forestgreen]
BOX_THICKNESS = 2.0

function panel_kw(;
        title  = "",
        xlabel = "",
        ylabel = "",
        xlims  = nothing,
        ylims  = nothing,
        bg     = :white,
)
    kw = (
        title                   = title,
        xlabel                  = xlabel,
        ylabel                  = ylabel,
        framestyle              = :box,
        grid                    = false,
        minorticks              = true,
        legend                  = :best,
        titlefont               = font(TITLE_SIZE, "Helvetica Bold"),
        guidefont               = font(LABEL_SIZE, "Helvetica Bold"),
        tickfont                = font(TICK_SIZE,  "Helvetica Bold"),
        legendfont              = font(LEGEND_SIZE,"Helvetica Bold"),
        tick_direction = :out,
        tick_length    = 10,
        left_margin             = 8mm,
        right_margin            = 4mm,
        top_margin              = 5mm,
        bottom_margin           = 8mm,
        foreground_color_border = :black,
        foreground_color_axis   = :black,
        background_color        = bg,
        background_color_inside = bg,
        xguidefontcolor         = :black,
        yguidefontcolor         = :black,
        xtickfontcolor          = :black,
        ytickfontcolor          = :black,
    )
    xlims !== nothing && (kw = merge(kw, (xlims=xlims,)))
    ylims !== nothing && (kw = merge(kw, (ylims=ylims,)))
    return kw
end

function ticks_length!(p, idx; tl=0.03, mtl=0.015)
    xt_major = Plots.xticks(p)[idx][1]
    yt_major = Plots.yticks(p)[idx][1]

    xl = Plots.xlims(p[idx])
    yl = Plots.ylims(p[idx])
    sz = p.attr[:size]
    aspect = sz[1] / sz[2]

    xt_minor = Float64[]; yt_minor = Float64[]
    if length(xt_major) ≥ 2
        for i in 1:length(xt_major)-1
            append!(xt_minor, range(xt_major[i], xt_major[i+1], length=6)[2:5])
        end
    end
    if length(yt_major) ≥ 2
        for i in 1:length(yt_major)-1
            append!(yt_minor, range(yt_major[i], yt_major[i+1], length=6)[2:5])
        end
    end

    dx_maj = tl * (xl[2] - xl[1]);     dy_maj = tl * aspect * (yl[2] - yl[1])
    dx_min = mtl * (xl[2] - xl[1]);    dy_min = mtl * aspect * (yl[2] - yl[1])

    plot!(p[idx], [xt_major xt_major]', [fill(yl[1], length(xt_major)) fill(yl[1]+dy_maj, length(xt_major))]';
          c=:black, lw=2.5, label=false)
    plot!(p[idx], [fill(xl[1], length(yt_major)) fill(xl[1]+dx_maj, length(yt_major))]', [yt_major yt_major]';
          c=:black, lw=2.5, label=false)

    if !isempty(xt_minor)
        plot!(p[idx], [xt_minor xt_minor]', [fill(yl[1], length(xt_minor)) fill(yl[1]+dy_min, length(xt_minor))]';
              c=:black, lw=2.0, label=false)
    end
    if !isempty(yt_minor)
        plot!(p[idx], [fill(xl[1], length(yt_minor)) fill(xl[1]+dx_min, length(yt_minor))]', [yt_minor yt_minor]';
              c=:black, lw=2.0, label=false)
    end

    plot!(p[idx], xlims=xl, ylims=yl)
end

# =======================================================
# 1. TRAJECTORY
# =======================================================
function get_global_traj_ylims(traj_file, configs, trials, node)
    ymin, ymax = Inf, -Inf
    h5open(traj_file, "r") do file
        for c in configs, t_key in trials
            path = "$(c.net)/$(c.param)/$(c.ic)/$t_key"
            haskey(file, path) || continue
            d    = read(file[path])
            traj = size(d,1) < size(d,2) ? d[node,:] : d[:,node]
            ymin = min(ymin, minimum(traj))
            ymax = max(ymax, maximum(traj))
        end
    end
    pad = (ymax - ymin) * 0.12
    return (ymin - pad, ymax + pad)
end

function fill_traj_panel!(p, idx, traj_file, net, param, ic,
                            trials, node, ylims_traj)
    first_series = true
    h5open(traj_file, "r") do file
        base = "$net/$param/$ic"
        haskey(file, base) || return
        for (i, tk) in enumerate(trials)
            full = "$base/$tk"
            haskey(file, full) || continue
            d    = read(file[full])
            traj = size(d,1) < size(d,2) ? d[node,:] : d[:,node]
            t    = collect(range(0.0, 2000.0, length=length(traj)))
            lbl  = i == 1 ? "Cell A" : "Cell B"
            if first_series
                plot!(p[idx], t, traj;
                    label  = lbl,
                    color  = TRAJ_COLORS[i],
                    lw     = LINE_W_TRAJ,
                    panel_kw(
                        xlabel = "Time (A.U.)",
                        ylabel = "X$(node) Level",
                        xlims  = TRAJ_XLIMS,
                        ylims  = ylims_traj,
                    )...
                )
                first_series = false
            else
                plot!(p[idx], t, traj;
                    label = lbl,
                    color = TRAJ_COLORS[i],
                    lw    = LINE_W_TRAJ,
                )
            end
        end
    end
end

# =======================================================
# 2. DISTRIBUTION -- dot/scatter of raw trial endpoints,
# =======================================================
function fit_gmm_1d(data_vec::Vector{Float64})
    d = reshape(data_vec, :, 1)
    gmm1 = sklearn.GaussianMixture(n_components=1, covariance_type="full", random_state=0).fit(d)
    gmm2 = sklearn.GaussianMixture(n_components=2, covariance_type="full", random_state=0).fit(d)
    return gmm1.bic(d) <= gmm2.bic(d) ? (ok=false, gmm=nothing) : (ok=true, gmm=gmm2)
end

function fit_gmm_2d(data_mat)
    d = Matrix(data_mat)
    gmm1 = sklearn.GaussianMixture(n_components=1, covariance_type="full", random_state=0).fit(d)
    gmm2 = sklearn.GaussianMixture(n_components=2, covariance_type="full", random_state=0).fit(d)
    return gmm1.bic(d) <= gmm2.bic(d) ? (ok=false, gmm=nothing) : (ok=true, gmm=gmm2)
end

on_component_1d(gmm) = (vec(gmm.means_)[1] >= vec(gmm.means_)[2]) ? 0 : 1
function on_component_2d(gmm)
    means = gmm.means_
    return sum(means[1, :]) >= sum(means[2, :]) ? 0 : 1
end

"""
    parse_accepted_nodes(str)

"x1,x2" -> [1,2]; "x2" -> [2]
"""
parse_accepted_nodes(str) = [parse(Int, replace(s, "x"=>"")) for s in split(str, ",")]

"""
    get_accepted_nodes(frac_file, hist_file, net, param)

Tries the `accepted_nodes` attribute (written by run_selected_net_1.jl) in
frac_file first, then hist_file. If neither has it, falls back to
inferring the active species directly from the raw trial data.
"""
function get_accepted_nodes(frac_file, hist_file, net, param)
    for file in (frac_file, hist_file)
        isfile(file) || continue
        result = h5open(file, "r") do f
            for key in ("$net/$param", "$net/$param/ic_1")
                haskey(f, key) || continue
                a = HDF5.attrs(f[key])
                haskey(a, "accepted_nodes") && return parse_accepted_nodes(a["accepted_nodes"])
            end
            return nothing
        end
        result !== nothing && return result
    end

    @warn "accepted_nodes attribute not found for $net/$param in frac_file or hist_file -- inferring from raw data instead"
    return infer_accepted_nodes(hist_file, net, param)
end

"""
    infer_accepted_nodes(hist_file, net, param; var_tol=1e-8)

Fallback when `accepted_nodes` isn't stored anywhere: reads final-time
trial values for all 3 species, keeps species with variance above
`var_tol` as "active", and if all 3 are active picks the best-BIC pair.
"""
function infer_accepted_nodes(hist_file, net, param; var_tol=1e-8)
    vals = h5open(hist_file, "r") do file
        g = file["$net/$param"]
        d = Dict(s => Float64[] for s in 1:3)
        for ik in keys(g)
            startswith(ik, "ic_") || continue
            for tk in keys(g[ik])
                startswith(tk, "trial_") || continue
                dat = read(g[ik][tk])
                for s in 1:3
                    push!(d[s], dat[s, end])
                end
            end
        end
        d
    end

    active = [s for s in 1:3 if var(vals[s]) > var_tol]
    isempty(active) && error("No active species detected for $net/$param -- check hist_file data")
    length(active) <= 2 && return active

    pairs = [(active[1],active[2]), (active[1],active[3]), (active[2],active[3])]
    best_pair, best_bic = pairs[1], Inf
    for (a,b) in pairs
        fit_res = fit_gmm_2d(hcat(vals[a], vals[b]))
        if fit_res.ok
            bic = fit_res.gmm.bic(hcat(vals[a], vals[b]))
            if bic < best_bic
                best_bic, best_pair = bic, (a,b)
            end
        end
    end
    return [best_pair[1], best_pair[2]]
end

"""
    time_to_col(ncols, t_target; t_max=2000.0)

Converts a TARGET TIME to the nearest column for an array with `ncols`
saved points spanning [0, t_max]. Computed per array/file rather than
assuming a fixed column index, since different files are saved at
different resolutions.
"""
function time_to_col(ncols::Int, t_target::Real; t_max::Real=2000.0)
    ncols <= 1 && return 1
    dt = t_max / (ncols - 1)
    return clamp(round(Int, t_target / dt) + 1, 1, ncols)
end

"""
    net_param_exists(file, net, param)

True if `net/param` is a valid group in the HDF5 file (and the file
itself exists).
"""
function net_param_exists(file, net, param)
    isfile(file) || return false
    h5open(file, "r") do f
        haskey(f, "$net/$param")
    end
end

"""
    find_trial_datasets(g)

Finds every 2D dataset reachable from group `g`, either directly inside
it or one level deeper (inside a subgroup). Doesn't assume a fixed
"ic_X/trial_Y" naming convention.
"""
function find_trial_datasets(g)
    paths = String[]
    for k in keys(g)
        obj = g[k]
        if obj isa HDF5.Dataset
            ndims(obj) == 2 && push!(paths, k)
        elseif obj isa HDF5.Group
            for kk in keys(obj)
                obj2 = obj[kk]
                if obj2 isa HDF5.Dataset && ndims(obj2) == 2
                    push!(paths, "$k/$kk")
                end
            end
        end
    end
    return paths
end

"""
    collect_trial_values(hist_files, net, param, sp, time_spec)

Reads trial-level raw values at the timepoint `time_spec` for species `sp`
from real per-trial datasets (found via `find_trial_datasets`), pooling
across every file in `hist_files` that has this net/param.
"""
function collect_trial_values(hist_files, net, param, sp, time_spec)
    vals = Dict(s => Float64[] for s in sp)
    for hist_file in hist_files
        isfile(hist_file) || continue
        h5open(hist_file, "r") do file
            key = "$net/$param"
            haskey(file, key) || return
            g = file[key]
            trial_paths = find_trial_datasets(g)
            for tpath in trial_paths
                dat = read(g[tpath])
                ncols = size(dat, 2)
                col = time_spec === :end ? ncols : time_to_col(ncols, time_spec)
                for s in sp
                    push!(vals[s], dat[s, col])
                end
            end
        end
    end
    return vals
end

"""
    fill_dot_panel!(p, idx, hist_files, net, param, sp, time_spec, title_str; bg=:white)

Draws a scatter (2 species) or histogram (1 species) of real per-trial
values, colored by state -- density-shaded, with a decision-boundary
contour for the 2-species case. Used for both IC and op-IC columns.
"""
function fill_dot_panel!(p, idx, hist_files, net, param, sp, time_spec, title_str; bg=:white)
    vals = collect_trial_values(hist_files, net, param, sp, time_spec)

    n_pts = isempty(sp) ? 0 : length(vals[sp[1]])
    if n_pts < 2
        @warn "fill_dot_panel!: fewer than 2 points found for $net/$param (time_spec=$time_spec, hist_files=$hist_files) -- skipping this panel" n_pts
        plot!(p[idx]; xlims=(0,1), ylims=(0,1), panel_kw(title=title_str, bg=bg)...)
        annotate!(p[idx], 0.5, 0.5, text("No data", 14, :gray))
        return
    end

    if length(sp) == 1
        sp_a = sp[1]
        xa = vals[sp_a]
        fit = fit_gmm_1d(xa)
        common = panel_kw(title=title_str, xlabel="X$(sp_a) Level", ylabel="Density", bg=bg)
        if !fit.ok
            histogram!(p[idx], xa; bins=40, normalize=:pdf, color=COLOR_GREY, alpha=ALPHA_UNI, label="Unimodal", common...)
        else
            on_comp = on_component_1d(fit.gmm)
            labels  = fit.gmm.predict(reshape(xa, :, 1))
            state2  = labels .== on_comp
            histogram!(p[idx], xa[.!state2]; bins=40, normalize=:pdf, color=COLOR_BI_1, alpha=ALPHA_BI, label="State 1", common...)
            histogram!(p[idx], xa[state2];   bins=40, normalize=:pdf, color=COLOR_BI_2, alpha=ALPHA_BI, label="State 2")
        end
        return
    end

    sp_a, sp_b = sp[1], sp[2]
    xa, xb = vals[sp_a], vals[sp_b]
    fit = fit_gmm_2d(hcat(xa, xb))
    common = panel_kw(title=title_str, xlabel="X$(sp_a) Level", ylabel="X$(sp_b) Level", bg=bg)

    if !fit.ok
        scatter!(p[idx], xa, xb; color=COLOR_GREY, markersize=5, markerstrokewidth=0.3,
                 markerstrokecolor=:white, label="Unimodal", common...)
        return
    end

    on_comp = on_component_2d(fit.gmm)
    proba   = fit.gmm.predict_proba(hcat(xa, xb))
    state2  = proba[:, on_comp+1] .>= 0.5

    nbins = 20
    xedges = range(minimum(xa) - 1e-9, maximum(xa) + 1e-9, length=nbins+1)
    yedges = range(minimum(xb) - 1e-9, maximum(xb) + 1e-9, length=nbins+1)
    bin_idx(v, edges) = clamp(searchsortedlast(edges, v), 1, length(edges)-1)

    counts1 = zeros(Int, nbins, nbins); counts2 = zeros(Int, nbins, nbins)
    for i in eachindex(xa)
        xi, yi = bin_idx(xa[i], xedges), bin_idx(xb[i], yedges)
        state2[i] ? (counts2[xi, yi] += 1) : (counts1[xi, yi] += 1)
    end
    density = zeros(Int, length(xa))
    for i in eachindex(xa)
        xi, yi = bin_idx(xa[i], xedges), bin_idx(xb[i], yedges)
        density[i] = state2[i] ? counts2[xi, yi] : counts1[xi, yi]
    end

    scatter!(p[idx], xa[.!state2], xb[.!state2];
             marker_z = density[.!state2], color = cgrad([:deepskyblue, :navy]),
             markersize = 5, markerstrokewidth = 0.3, markerstrokecolor = :black,
             colorbar = false, label = "State 1", common...)
    scatter!(p[idx], xa[state2], xb[state2];
             marker_z = density[state2], color = cgrad([:gold, :darkorange]),
             markersize = 5, markerstrokewidth = 0.3, markerstrokecolor = :black,
             colorbar = false, label = "State 2")

    xr = range(minimum(xa), maximum(xa), length=150)
    yr = range(minimum(xb), maximum(xb), length=150)
    grid_pts = hcat([x for x in xr for y in yr], [y for x in xr for y in yr])
    proba_grid = fit.gmm.predict_proba(grid_pts)[:, on_comp+1]
    proba_mat  = reshape(proba_grid, length(yr), length(xr))
    contour!(p[idx], xr, yr, proba_mat; levels=[0.5], linewidth=3, color=:black, label="Decision boundary")
end

# =======================================================
# 3. FALLBACK: on/off fraction BAR (only used if a net/param turns out
#    to have no raw trial data even in the op-IC hist file).
# =======================================================
function fill_fraction_bar_panel!(p, idx, frac_file, net, param, t_target, title_str; bg=:white)
    if !isfile(frac_file)
        @warn "frac_file not found -- skipping panel" frac_file
        plot!(p[idx]; xlims=(0,1), ylims=(0,1), panel_kw(title=title_str, bg=bg)...)
        annotate!(p[idx], 0.5, 0.5, text("No data", 14, :gray))
        return
    end

    on_val, off_val = h5open(frac_file, "r") do file
        key = "$net/$param"
        haskey(file, key) || return (nothing, nothing)
        g = file[key]
        haskey(g, "on_fraction") || return (nothing, nothing)
        on_frac  = read(g["on_fraction"])
        off_frac = haskey(g, "off_fraction") ? read(g["off_fraction"]) : (1.0 .- on_frac)
        n   = length(on_frac)
        col = t_target === :end ? n : time_to_col(n, t_target)
        (Float64(on_frac[col]), Float64(off_frac[col]))
    end

    if on_val === nothing
        @warn "on_fraction not found for $net/$param in $frac_file -- skipping panel"
        plot!(p[idx]; xlims=(0,1), ylims=(0,1), panel_kw(title=title_str, bg=bg)...)
        annotate!(p[idx], 0.5, 0.5, text("No data", 14, :gray))
        return
    end

    common = panel_kw(title=title_str, xlabel="", ylabel="Fraction of population", bg=bg,
                       xlims=(0.3, 2.7), ylims=(0.0, 1.0))
    bar!(p[idx], [1, 2], [off_val, on_val];
         xticks = ([1, 2], ["Off", "On"]),
         color  = [COLOR_BI_1, COLOR_BI_2],
         legend = false, common...)
end

"""
    fill_opic_panel!(p, idx, opic_hist_file, opic_frac_file, net, param, sp, t_target, title_str; bg=:white)

Tries a real scatter/histogram from `opic_hist_file` (raw per-trial data)
first. If that net/param has fewer than 2 points there (e.g. missing),
falls back to the on/off fraction bar from `opic_frac_file` instead of
showing nothing.
"""
function fill_opic_panel!(p, idx, opic_hist_file, opic_frac_file, net, param, sp, t_target, title_str; bg=:white)
    vals = collect_trial_values([opic_hist_file], net, param, sp, t_target)
    n_pts = isempty(sp) ? 0 : length(vals[sp[1]])
    if n_pts >= 2
        fill_dot_panel!(p, idx, [opic_hist_file], net, param, sp, t_target, title_str; bg=bg)
    else
        @warn "No raw trial data found in $opic_hist_file for $net/$param at t=$t_target -- falling back to on/off fraction bar from $opic_frac_file" n_pts
        fill_fraction_bar_panel!(p, idx, opic_frac_file, net, param, t_target, title_str; bg=bg)
    end
end

# =======================================================
# 4. COMBINED FIGURE -- 5 columns per row:
#    1: trajectory 
#    2: IC scatter at t=50
#    3: IC scatter at t=2000
#    4: opposite-IC scatter  at t=50
#    5: opposite-IC scatter at t=2000
# =======================================================
function build_combined_3x3(configs, traj_file, hist_file, frac_file,
                              opic_dir, opic_hist_file_for_net, opic_frac_file_for_net,
                              node; savepath="combined_3x3.svg")

    n_rows = length(configs)
    n_cols = 5

    ylims_traj = get_global_traj_ylims(traj_file, configs, target_trials, node)

    p = plot(
        layout           = (n_rows, n_cols),
        size             = (600*n_cols, 550*n_rows),
        dpi              = 600,
        background_color = :white,
    )

    for (row, c) in enumerate(configs)
        idx_traj       = (row-1)*n_cols + 1
        idx_ic_early   = (row-1)*n_cols + 2
        idx_ic_final   = (row-1)*n_cols + 3
        idx_opic_early = (row-1)*n_cols + 4
        idx_opic_final = (row-1)*n_cols + 5

        t_ic_early   = (row == 1) ? "IC: Time = 50"       : ""
        t_ic_final   = (row == 1) ? "IC: Time = 2000"     : ""
        t_opic_early = (row == 1) ? "Op IC: Time = 50"    : ""
        t_opic_final = (row == 1) ? "Op IC: Time = 2000"  : ""

        # Column 1: Trajectories (raw scale) -- current IC only, from v3 file
        fill_traj_panel!(p, idx_traj, traj_file, c.net, c.param, c.ic,
                         target_trials, node, ylims_traj)

        # Which species were accepted for this net/param (read, not recomputed)
        sp = get_accepted_nodes(frac_file, hist_file, c.net, c.param)

        # This network's opposite-IC files (raw trial data + fraction fallback)
        haskey(opic_hist_file_for_net, c.net) ||
            error("No opposite-IC hist file mapping for $(c.net) -- add it to opic_hist_file_for_net")
        haskey(opic_frac_file_for_net, c.net) ||
            error("No opposite-IC fraction file mapping for $(c.net) -- add it to opic_frac_file_for_net")
        hist_file_opic = joinpath(opic_dir, opic_hist_file_for_net[c.net])
        frac_file_opic = joinpath(opic_dir, opic_frac_file_for_net[c.net])

        # Columns 2-3: IC scatter at t=50 and t=2000 (real per-trial data)
        fill_dot_panel!(p, idx_ic_early, [hist_file], c.net, c.param, sp, 50.0, t_ic_early; bg=:white)
        fill_dot_panel!(p, idx_ic_final, [hist_file], c.net, c.param, sp, :end, t_ic_final; bg=:white)

        # Columns 4-5: Opposite-IC scatter at t=50 and t=2000 (real per-trial
        # data from opic_hist_file, falling back to the fraction bar only
        # if that net/param has no raw data there)
        fill_opic_panel!(p, idx_opic_early, hist_file_opic, frac_file_opic, c.net, c.param, sp, 50.0, t_opic_early; bg=:white)
        fill_opic_panel!(p, idx_opic_final, hist_file_opic, frac_file_opic, c.net, c.param, sp, :end, t_opic_final; bg=:white)

        ticks_length!(p, idx_traj;     tl=0.035, mtl=0.015)
        ticks_length!(p, idx_ic_early; tl=0.035, mtl=0.015)
        ticks_length!(p, idx_ic_final; tl=0.035, mtl=0.015)
    end

    display(p)
    #savefig(p, savepath)
    return p
end

# =======================================================
# EXECUTION
# =======================================================

# Keep only configs whose net/param pair exists in the current-IC file AND
# in at least one of that network's opposite-IC files (raw hist or the
# fraction-summary fallback).
valid_configs = []
for c in configs
    if !haskey(opic_hist_file_for_net, c.net) || !haskey(opic_frac_file_for_net, c.net)
        @warn "Skipping $(c.net)/$(c.param): missing opposite-IC file mapping"
        continue
    end
    hist_file_opic_check = joinpath(OPIC_DIR, opic_hist_file_for_net[c.net])
    frac_file_opic_check = joinpath(OPIC_DIR, opic_frac_file_for_net[c.net])

    in_ic         = net_param_exists(hist_file, c.net, c.param)
    in_opic_hist  = net_param_exists(hist_file_opic_check, c.net, c.param)
    in_opic_frac  = net_param_exists(frac_file_opic_check, c.net, c.param)

    if in_ic && (in_opic_hist || in_opic_frac)
        push!(valid_configs, c)
    else
        @warn "Skipping $(c.net)/$(c.param): in IC file? $in_ic -- in op-IC hist file? $in_opic_hist -- in op-IC fraction file? $in_opic_frac"
    end
end

isempty(valid_configs) && error("No net/param pairs found in the current-IC file with a matching opposite-IC source -- nothing to plot.")

p_out = build_combined_3x3(valid_configs, traj_file, hist_file, frac_file,
                            OPIC_DIR, opic_hist_file_for_net, opic_frac_file_for_net, node_idx)
println("\n=========== ANALYSIS COMPLETE ===========")

### Step 1 : Network validity filter + op_ic row filter

Two independent filters, both applied:
1. **Network-level** -- `is_null_or_disjoint()` (active-node union-find over the adjacency
   matrix, matching the logic in `run_selected_net_1.jl`) builds the valid-network universe

2. **Row-level** -- `summary_clean_noise1.csv` is inner-joined against
   `summary_clean_noise1_op_ic.csv` on `(net_id, param_id)`, keeping only rows confirmed under
   both ICs, saved as `summary_clean_noise1_filtered.csv` for every downstream analysis cell.

In [ ]:
###############################################################
# Step 1a: valid-network filter
###############################################################

using JLD2

"""
    active_node_indices(mat)

Indices of nodes that have at least one nonzero edge in or out.
"""
function active_node_indices(mat)
    active_mask = [any(mat[i,:] .!= 0) || any(mat[:,i] .!= 0) for i in 1:3]
    return findall(active_mask)
end

"""
    find_components(active_indices, mat)

Union-find over the active nodes only: two active nodes are in the same
component if there's an edge between them in either direction. Returns
one Vector{Int} of node indices per connected component.
"""
function find_components(active_indices::Vector{Int}, mat)
    parent = Dict(idx => idx for idx in active_indices)

    function find_root(x)
        while parent[x] != x
            parent[x] = parent[parent[x]]
            x = parent[x]
        end
        return x
    end

    function union_nodes!(x, y)
        rx, ry = find_root(x), find_root(y)
        if rx != ry
            parent[rx] = ry
        end
    end

    for i in active_indices, j in active_indices
        if i < j && (mat[i,j] != 0 || mat[j,i] != 0)
            union_nodes!(i, j)
        end
    end

    comps = Dict{Int,Vector{Int}}()
    for idx in active_indices
        r = find_root(idx)
        push!(get!(comps, r, Int[]), idx)
    end

    return collect(values(comps))
end

"""
    is_null_or_disjoint(mat)

True (excluded) if `mat` is all-zero, or if the active nodes split into
more than one connected component. A single active node (e.g. one
self-loop) is not disjoint -- there's nothing for it to be disconnected from.
"""
function is_null_or_disjoint(mat)
    all(mat .== 0) && return true
    active_idx = active_node_indices(mat)
    isempty(active_idx) && return true
    comps = find_components(active_idx, mat)
    return length(comps) > 1
end

is_valid_network(mat) = !is_null_or_disjoint(mat)

# ------------------------------------------------------------
# Sanity check against the JLD2 network table
# ------------------------------------------------------------
dir_check          = raw"F:\HR_Project\noise1"
network_data_check = load(joinpath(dir_check, "unique_networks_table.jld2"))
all_matrices_check = network_data_check["network_table"]

n_total = length(all_matrices_check)
n_valid = sum(is_valid_network(all_matrices_check[id].adj_matrix) for id in keys(all_matrices_check))

println("Total networks in JLD2 table   : ", n_total)
println("Valid (non-null, non-disjoint) : ", n_valid, "  (should be 3325)")

###############################################################
# Step 1b: filter noise1 summary to rows also present in op_ic summary
###############################################################

using CSV
using DataFrames

main_csv     = raw"F:\HR_Project\noise1\summary_clean_noise1.csv"
op_ic_csv    = raw"F:\HR_Project\noise1_op_ic\summary_clean_noise1_op_ic.csv"
filtered_csv = raw"F:\HR_Project\noise1\summary_clean_noise1_filtered.csv"

df_main = CSV.read(main_csv, DataFrame; header=true)
rename!(df_main, names(df_main)[1] => :net_id)

df_opic = CSV.read(op_ic_csv, DataFrame; header=true)
rename!(df_opic, names(df_opic)[1] => :net_id)

# Unique (net_id, param_id) pairs present in the op_ic file
opic_keys = unique(df_opic[:, [:net_id, :param_id]])

# Keep only main rows whose (net_id, param_id) also exists in op_ic
df_filtered = innerjoin(df_main, opic_keys, on = [:net_id, :param_id])

println("\nMain rows             : ", nrow(df_main))
println("Rows matched to op_ic  : ", nrow(df_filtered))
println("Rows dropped           : ", nrow(df_main) - nrow(df_filtered))
println("Unique networks before : ", length(unique(df_main.net_id)))
println("Unique networks after  : ", length(unique(df_filtered.net_id)))

CSV.write(filtered_csv, df_filtered)
println("Saved filtered summary to: ", filtered_csv)


### Shared utilities for Fig 2C–E: contingency heatmap + %-bar plotting

In [ ]:
###############################################################
# Shared plotting utilities for Fig 2C, 2D, 2E
###############################################################

using Plots
using Colors

Plots.default(dpi=600)

"""
    plot_contingency_heatmap(a, b, c, d; with_label, without_label, p_val_str=nothing, odds_ratio=nothing)

2x2 log10(count+1) heatmap in the C/D/E style. `a` = HR & with-feature,
`b` = HR & without-feature, `c` = Non-HR & with-feature, `d` = Non-HR &
without-feature -- exactly the a/b/c/d each Fisher-analysis cell prints.
Pass the odds_ratio / p-value straight from that cell too (e.g. `OR`, `p_two`)
so the annotation on the plot can never drift from the numbers behind it.
"""
function plot_contingency_heatmap(a, b, c, d; with_label="With Feature", without_label="Without Feature",
                                   p_val_str=nothing, odds_ratio=nothing)

    counts = [a b; c d]
    log_counts = log10.(counts .+ 1)
    x = [1, 2]; y = [1, 2]

    title_str = odds_ratio === nothing ? "" : "OR = " * string(round(odds_ratio, sigdigits=3))
    if p_val_str !== nothing
        title_str = title_str == "" ? "p $(p_val_str)" : "$title_str, p $(p_val_str)"
    end

    h = heatmap(
        x, y, log_counts,
        c = :YlGnBu, colorbar = true, colorbar_pad = 15,
        framestyle = :box, grid = false, linewidth = 2.0, aspect_ratio = 1, size = (750, 600),
        xlims = (0.5, 2.5), ylims = (0.5, 2.5),
        xticks = (x, [with_label, without_label]),
        yticks = (y, ["HR-Enabling \nNetworks", "HR-Disabling \nNetworks"]),
        yflip = true,
        xtickfont  = font(14, "Helvetica Bold"), ytickfont  = font(14, "Helvetica Bold"),
        xguidefont = font(18, "Helvetica Bold"), yguidefont = font(18, "Helvetica Bold"),
        legendfont = font(12, "Helvetica Bold"),
        left_margin = 25Plots.mm, right_margin = 20Plots.mm, top_margin = 15Plots.mm, bottom_margin = 15Plots.mm,
        colorbar_title = "\nlog10(count + 1)",
        title = title_str,
    )

    annotate!(1, 1, text("n = $a", 12, a > 400 ? :white : :black, :bold))
    annotate!(2, 1, text("n = $b", 12, b > 400 ? :white : :black, :bold))
    annotate!(1, 2, text("n = $c", 12, c > 400 ? :white : :black, :bold))
    annotate!(2, 2, text("n = $d", 12, d > 400 ? :white : :black, :bold))

    return h
end

"""
    plot_pct_bar(a, b, c, d)

0-100% horizontal stacked bar (HR row on top, Non-HR row on bottom) showing
the fraction of networks with the feature, using the SAME a/b/c/d as
`plot_contingency_heatmap` -- so the bar and the table can never disagree.
"""
function plot_pct_bar(a, b, c, d)
    hr_total    = a + b
    nonhr_total = c + d
    hr_pct      = hr_total    == 0 ? 0.0 : (a / hr_total)    * 100
    nonhr_pct   = nonhr_total == 0 ? 0.0 : (c / nonhr_total) * 100

    col_with    = colorant"#F08080"  # w/ feature (red)
    col_without = colorant"#808080"  # w/o feature (gray)
    bar_h, y_hr, y_nonhr = 0.6, 2.0, 1.0

    hbar_segment(y, x0, x1, h) = Shape([x0, x1, x1, x0], [y-h/2, y-h/2, y+h/2, y+h/2])

    p = plot(
        legend = false, xlims = (0, 100), ylims = (0.3, 2.7),
        yticks = ([y_nonhr, y_hr], ["Non-HR", "HR"]),
        xticks = ([0, 100], ["0%", "100%"]),
        framestyle = :box, grid = false, size = (500, 400),
        xtickfont = font(14, "Helvetica"), ytickfont = font(14, "Helvetica"),
        bottom_margin = 5Plots.mm, left_margin = 8Plots.mm, right_margin = 8Plots.mm,
        xmirror = true,
    )

    plot!(p, hbar_segment(y_hr, 0, hr_pct, bar_h), color=col_with, linecolor=:black, linewidth=0.5)
    plot!(p, hbar_segment(y_hr, hr_pct, 100, bar_h), color=col_without, linecolor=:black, linewidth=0.5)
    plot!(p, hbar_segment(y_nonhr, 0, nonhr_pct, bar_h), color=col_with, linecolor=:black, linewidth=0.5)
    plot!(p, hbar_segment(y_nonhr, nonhr_pct, 100, bar_h), color=col_without, linecolor=:black, linewidth=0.5)

    return p
end

println("Shared C/D/E utilities loaded: plot_contingency_heatmap(), plot_pct_bar()")


### Fig 2C : Contingency table & hist for FB, w/o FB

In [ ]:
###############################################################
# Full Analysis: Multimodality vs Feedback Fisher's Test , OR
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using Plots
using HypothesisTests

println("Starting analysis...")

# ------------------------------------------------------------
# 1. Load multimodal CSV
# ------------------------------------------------------------
csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1_filtered.csv"
df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

println("Loaded multimodal data.")

# ------------------------------------------------------------
# 2. Load all networks from JLD2
# ------------------------------------------------------------
dir = raw"F:\HR_Project\noise1"
jld_path = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]
all_net_ids = collect(keys(all_matrices))

println("Loaded network matrices.")

# ------------------------------------------------------------
# 3. Function to detect feedback
# ------------------------------------------------------------
function has_feedback(mat)
    # 1. Self-loops
    for i in 1:3
        if mat[i,i] != 0
            return true
        end
    end
    # 2. Pairwise loops
    pairs = [(1,2),(1,3),(2,3)]
    for (i,j) in pairs
        if mat[i,j] != 0 && mat[j,i] != 0
            return true
        end
    end
    # 3. 3-node cycles
    clockwise = mat[1,2]!=0 && mat[2,3]!=0 && mat[3,1]!=0
    counter = mat[1,3]!=0 && mat[3,2]!=0 && mat[2,1]!=0
    if clockwise || counter
        return true
    end
    return false
end

# ------------------------------------------------------------
# 4. Filter out networks that are all-zero
# ------------------------------------------------------------
filtered_ids = [id for id in all_net_ids if is_valid_network(all_matrices[id].adj_matrix)]
all_nets_df = DataFrame(net_id = filtered_ids)

# Merge with CSV counts
net_counts_full = leftjoin(all_nets_df, net_counts, on=:net_id)

# Replace missing counts and percentages with 0
net_counts_full.count = coalesce.(net_counts_full.count, 0)
net_counts_full.percentage_multimodal = coalesce.(net_counts_full.percentage_multimodal, 0.0)

# Add feedback info
net_counts_full.feedback = [has_feedback(all_matrices[id].adj_matrix) for id in net_counts_full.net_id]

# ------------------------------------------------------------
# 5. Compute summary counts
# ------------------------------------------------------------
n_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== true))
n_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== false))
n_non_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== true))
n_non_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== false))

println("\nContingency Table Counts:")
println("Multimodal + FB      = ", n_multimodal_with_fb)
println("Multimodal + No FB   = ", n_multimodal_no_fb)
println("Non-Multimodal + FB  = ", n_non_multimodal_with_fb)
println("Non-Multimodal + NoFB= ", n_non_multimodal_no_fb)

# ------------------------------------------------------------
# 6. Build 2x2 Table
# ------------------------------------------------------------
a = n_multimodal_with_fb
b = n_multimodal_no_fb
c = n_non_multimodal_with_fb
d = n_non_multimodal_no_fb

table = [a b; c d]
println("\n2x2 Table:")
println(table)

# ------------------------------------------------------------
# 7. Odds Ratio (Haldane-Anscombe correction if zero)
# ------------------------------------------------------------
# Original counts (integers)
a_int = n_multimodal_with_fb
b_int = n_multimodal_no_fb
c_int = n_non_multimodal_with_fb
d_int = n_non_multimodal_no_fb

# Odds ratio (apply 0.5 correction if zero) Haldane-Anscombe correction
a_or, b_or, c_or, d_or = a_int, b_int, c_int, d_int
if any([a_int,b_int,c_int,d_int] .== 0)
    a_or += 0.5
    b_or += 0.5
    c_or += 0.5
    d_or += 0.5
end

OR = (a_or*d_or)/(b_or*c_or)
logOR = log(OR)

println("\nOdds Ratio = ", OR)
println("Log Odds Ratio = ", logOR)

# Fisher test (use integer counts!)
fet = FisherExactTest(a_int, b_int, c_int, d_int)
p_two = pvalue(fet)               # two-sided
p_one = pvalue(fet, tail=:right)  # one-sided

println("\nFisher Test Results:")
println("One-sided p-value  = ", p_one)
println("Two-sided p-value  = ", p_two)

# ------------------------------------------------------------
# 9. Separate percentages for plotting
# ------------------------------------------------------------
with_fb = net_counts_full.percentage_multimodal[net_counts_full.feedback .== true]
without_fb = net_counts_full.percentage_multimodal[net_counts_full.feedback .== false]

# ------------------------------------------------------------
# 10. Plot stacked histogram
# ------------------------------------------------------------
p_hist_C = histogram(
    [with_fb, without_fb],
    label = ["With Feedback" "Without Feedback"],
    color = [:teal :orange],
    bins = 30,
    title = "Overall Network Multimodality",
    xlabel = "Percentage Multimodal",
    ylabel = "Number of Networks",
    legend = :topright,
    alpha = 0.7,
    lw = 0.5,
    xlims = (0, 100),
    ylims = (0, 3000),
    normalize = false
)

println("\nAnalysis complete.")
display(p_hist_C)

In [ ]:
# Fig 2C heatmap, built from the Fisher-test contingency table computed above
h_C = plot_contingency_heatmap(a, b, c, d;
        with_label="With FB", without_label="Without FB",
        p_val_str = p_two < 0.0001 ? "< 0.0001" : "= " * string(round(p_two, sigdigits=4)),
        odds_ratio = OR)
display(h_C)
# savefig(h_C, "contingency_heatmap_FB.png")
# savefig(h_C, "contingency_heatmap_FB.svg")


In [ ]:
# Fig 2C distribution bar, using the same a,b,c,d as the heatmap above
p_C_bar = plot_pct_bar(a, b, c, d)
display(p_C_bar)
#savefig(p_C_bar, "figure_2C_style.png")


### Fig 2D : Contingency table & hist for PFB, w/o PFB

In [ ]:
###############################################################
# Full Analysis: Multimodality vs Positive Feedback Fisher's Test , OR
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using Plots
using HypothesisTests
using Colors

gr()
Plots.default(dpi=600)

println("Starting analysis...")

# ------------------------------------------------------------
# 1. Load multimodal CSV
# ------------------------------------------------------------
csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1_filtered.csv"
df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

println("Loaded multimodal data.")

# ------------------------------------------------------------
# 2. Load all networks from JLD2
# ------------------------------------------------------------
dir = raw"F:\HR_Project\noise1"
jld_path = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]
all_net_ids = collect(keys(all_matrices))

println("Loaded network matrices.")

# ------------------------------------------------------------
# 3. Function to detect positive feedback loops specifically
# ------------------------------------------------------------
function has_positive_feedback(mat)
    # 1. Self-loops (must be auto-activation to be positive)
    for i in 1:3
        if mat[i,i] > 0
            return true
        end
    end
    # 2. Pairwise loops (product of signs must be positive: +/+ or -/-)
    pairs = [(1,2),(1,3),(2,3)]
    for (i,j) in pairs
        if mat[i,j] != 0 && mat[j,i] != 0
            if (mat[i,j] * mat[j,i]) > 0
                return true
            end
        end
    end
    # 3. 3-node cycles (product of all three signs must be positive)
    if mat[1,2] != 0 && mat[2,3] != 0 && mat[3,1] != 0
        if (mat[1,2] * mat[2,3] * mat[3,1]) > 0
            return true
        end
    end
    if mat[1,3] != 0 && mat[3,2] != 0 && mat[2,1] != 0
        if (mat[1,3] * mat[3,2] * mat[2,1]) > 0
            return true
        end
    end
    return false
end

# ------------------------------------------------------------
# 4. Filter out networks that are all-zero and merge
# ------------------------------------------------------------
filtered_ids = [id for id in all_net_ids if is_valid_network(all_matrices[id].adj_matrix)]
all_nets_df = DataFrame(net_id = filtered_ids)

# Merge with CSV counts
net_counts_full = leftjoin(all_nets_df, net_counts, on=:net_id)

# Replace missing counts and percentages with 0
net_counts_full.count = coalesce.(net_counts_full.count, 0)
net_counts_full.percentage_multimodal = coalesce.(net_counts_full.percentage_multimodal, 0.0)

# Add positive feedback info
net_counts_full.feedback = [has_positive_feedback(all_matrices[id].adj_matrix) for id in net_counts_full.net_id]

# ------------------------------------------------------------
# 5. Compute summary counts
# ------------------------------------------------------------
n_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== true))
n_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== false))
n_non_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== true))
n_non_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== false))

println("\nContingency Table Counts (Positive FB):")
println("Multimodal + Pos FB      = ", n_multimodal_with_fb)
println("Multimodal + No Pos FB   = ", n_multimodal_no_fb)
println("Non-Multimodal + Pos FB  = ", n_non_multimodal_with_fb)
println("Non-Multimodal + No PosFB= ", n_non_multimodal_no_fb)

# ------------------------------------------------------------
# 6. Build 2x2 Table
# ------------------------------------------------------------
a = n_multimodal_with_fb
b = n_multimodal_no_fb
c = n_non_multimodal_with_fb
d = n_non_multimodal_no_fb

table = [a b; c d]
println("\n2x2 Table:")
println(table)

# ------------------------------------------------------------
# 7. Odds Ratio (Haldane-Anscombe correction if zero)
# ------------------------------------------------------------
a_int = n_multimodal_with_fb
b_int = n_multimodal_no_fb
c_int = n_non_multimodal_with_fb
d_int = n_non_multimodal_no_fb

a_or, b_or, c_or, d_or = a_int, b_int, c_int, d_int
if any([a_int,b_int,c_int,d_int] .== 0)
    a_or += 0.5
    b_or += 0.5
    c_or += 0.5
    d_or += 0.5
end

OR = (a_or*d_or)/(b_or*c_or)
logOR = log(OR)

println("\nOdds Ratio = ", OR)
println("Log Odds Ratio = ", logOR)

# Fisher test (using integer counts)
fet = FisherExactTest(a_int, b_int, c_int, d_int)
p_two = pvalue(fet)               
p_one = pvalue(fet, tail=:right)  

println("\nFisher Test Results:")
println("One-sided p-value  = ", p_one)
println("Two-sided p-value  = ", p_two)

# ------------------------------------------------------------
# 8. Manual Horizontal Stacked Bar Plot 
#     HR = Multimodal, Non-HR = Non-Multimodal
#     Uses the Fisher's test contingency counts a, b, c, d
# ------------------------------------------------------------
hr_total    = a + b   # multimodal total
nonhr_total = c + d   # non-multimodal total

hr_pct_fbl    = hr_total    == 0 ? 0.0 : (a / hr_total)    * 100   # multimodal w/ pos FB
nonhr_pct_fbl = nonhr_total == 0 ? 0.0 : (c / nonhr_total) * 100   # non-multimodal w/ pos FB

col_fbl    = colorant"#F08080"  # w/ Pos FBL  (red)
col_no_fbl = colorant"#808080"  # w/o Pos FBL (gray)

bar_h   = 0.6   # bar thickness
y_hr    = 2.0   # y-center for HR (Multimodal) bar (top)
y_nonhr = 1.0   # y-center for Non-HR (Non-Multimodal) bar (bottom)

function hbar_segment(y, x0, x1, h)
    Shape([x0, x1, x1, x0], [y-h/2, y-h/2, y+h/2, y+h/2])
end

plot2 = plot(
    legend = false,
    xlims  = (0, 100),
    ylims  = (0.3, 2.7),
    yticks = ([y_nonhr, y_hr], ["Non-HR", "HR"]),
    xticks = ([0, 100], ["0%", "100%"]),
    framestyle = :box,
    grid = false,
    size = (500, 400),
    xtickfont  = font(14, "Helvetica"),
    ytickfont  = font(14, "Helvetica"),
    bottom_margin = 5Plots.mm,
    left_margin   = 8Plots.mm,
    right_margin   = 8Plots.mm,
    xmirror = true   
)

# HR (Multimodal) bar: red first ,then gray 
plot!(plot2, hbar_segment(y_hr, 0, hr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_hr, hr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

# Non-HR (Non-Multimodal) bar: red first, then gray on top
plot!(plot2, hbar_segment(y_nonhr, 0, nonhr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_nonhr, nonhr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

display(plot2)

# ------------------------------------------------------------
# 9. SAVE Horizontal Stacked Bar Plot
# ------------------------------------------------------------
# savefig(plot2, "figure_2D_style_PosFB.svg")
# savefig(plot2, "figure_2D_style_PosFB.png")

println("Analysis complete. Stacked horizontal bar plot saved.")

In [ ]:
# Fig 2D heatmap, built from the Positive-FB contingency table computed above
h_D = plot_contingency_heatmap(a, b, c, d;
        with_label="With PosFB", without_label="Without PosFB",
        p_val_str = p_two < 0.0001 ? "< 0.0001" : "= " * string(round(p_two, sigdigits=4)),
        odds_ratio = OR)
display(h_D)
# savefig(h_D, "contingency_heatmap_PosFB.png")
# savefig(h_D, "contingency_heatmap_PosFB.svg")


### Fig 2E : Contingency table & hist for NFB, w/o NFB

In [ ]:
###############################################################
# Full Analysis: Multimodality vs Negative Feedback Fisher's Test, OR
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using Plots
using HypothesisTests
using Colors

gr()
Plots.default(dpi=600)

println("Starting analysis...")

# ------------------------------------------------------------
# 1. Load multimodal CSV
# ------------------------------------------------------------
csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1_filtered.csv"
df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

println("Loaded multimodal data.")

# ------------------------------------------------------------
# 2. Load all networks from JLD2
# ------------------------------------------------------------
dir = raw"F:\HR_Project\noise1"
jld_path = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]
all_net_ids = collect(keys(all_matrices))

println("Loaded network matrices.")

# ------------------------------------------------------------
# 3. Function to detect negative feedback loops specifically
# ------------------------------------------------------------
function has_negative_feedback(mat)
    # 1. Self-loops (must be auto-inhibition to be negative)
    for i in 1:3
        if mat[i,i] < 0
            return true
        end
    end
    # 2. Pairwise loops (product of signs must be negative, e.g., +/- or -/+)
    pairs = [(1,2),(1,3),(2,3)]
    for (i,j) in pairs
        if mat[i,j] != 0 && mat[j,i] != 0
            if (mat[i,j] * mat[j,i]) < 0
                return true
            end
        end
    end
    # 3. 3-node cycles (product of all three signs must be negative)
    if mat[1,2] != 0 && mat[2,3] != 0 && mat[3,1] != 0
        if (mat[1,2] * mat[2,3] * mat[3,1]) < 0
            return true
        end
    end
    if mat[1,3] != 0 && mat[3,2] != 0 && mat[2,1] != 0
        if (mat[1,3] * mat[3,2] * mat[2,1]) < 0
            return true
        end
    end
    return false
end

# ------------------------------------------------------------
# 4. Filter out networks that are all-zero and merge
# ------------------------------------------------------------
filtered_ids = [id for id in all_net_ids if is_valid_network(all_matrices[id].adj_matrix)]
all_nets_df = DataFrame(net_id = filtered_ids)

# Merge with CSV counts
net_counts_full = leftjoin(all_nets_df, net_counts, on=:net_id)

# Replace missing counts and percentages with 0
net_counts_full.count = coalesce.(net_counts_full.count, 0)
net_counts_full.percentage_multimodal = coalesce.(net_counts_full.percentage_multimodal, 0.0)

# Add negative feedback info
net_counts_full.feedback = [has_negative_feedback(all_matrices[id].adj_matrix) for id in net_counts_full.net_id]

# ------------------------------------------------------------
# 5. Compute summary counts
# ------------------------------------------------------------
n_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== true))
n_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== false))
n_non_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== true))
n_non_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== false))

println("\nContingency Table Counts (Negative FB):")
println("Multimodal + Neg FB      = ", n_multimodal_with_fb)
println("Multimodal + No Neg FB   = ", n_multimodal_no_fb)
println("Non-Multimodal + Neg FB  = ", n_non_multimodal_with_fb)
println("Non-Multimodal + No NegFB= ", n_non_multimodal_no_fb)

# ------------------------------------------------------------
# 6. Build 2x2 Table
# ------------------------------------------------------------
a = n_multimodal_with_fb
b = n_multimodal_no_fb
c = n_non_multimodal_with_fb
d = n_non_multimodal_no_fb

table = [a b; c d]
println("\n2x2 Table:")
println(table)

# ------------------------------------------------------------
# 7. Odds Ratio (Haldane-Anscombe correction if zero)
# ------------------------------------------------------------
a_int = n_multimodal_with_fb
b_int = n_multimodal_no_fb
c_int = n_non_multimodal_with_fb
d_int = n_non_multimodal_no_fb

a_or, b_or, c_or, d_or = a_int, b_int, c_int, d_int
if any([a_int,b_int,c_int,d_int] .== 0)
    a_or += 0.5
    b_or += 0.5
    c_or += 0.5
    d_or += 0.5
end

OR = (a_or*d_or)/(b_or*c_or)
logOR = log(OR)

println("\nOdds Ratio = ", OR)
println("Log Odds Ratio = ", logOR)

# Fisher test (using integer counts)
fet = FisherExactTest(a_int, b_int, c_int, d_int)
p_two = pvalue(fet)               
p_one = pvalue(fet, tail=:right)  

println("\nFisher Test Results:")
println("One-sided p-value  = ", p_one)
println("Two-sided p-value  = ", p_two)

# ------------------------------------------------------------
# 8. Manual Horizontal Stacked Bar Plot 
#     HR = Multimodal, Non-HR = Non-Multimodal
#     Uses the Fisher's test contingency counts a, b, c, d
# ------------------------------------------------------------
hr_total    = a + b   # multimodal total
nonhr_total = c + d   # non-multimodal total

hr_pct_fbl    = hr_total    == 0 ? 0.0 : (a / hr_total)    * 100   # multimodal w/ neg FB
nonhr_pct_fbl = nonhr_total == 0 ? 0.0 : (c / nonhr_total) * 100   # non-multimodal w/ neg FB

col_fbl    = colorant"#F08080"  # w/ Neg FBL  (red)
col_no_fbl = colorant"#808080"  # w/o Neg FBL (gray)

bar_h   = 0.6   # bar thickness
y_hr    = 2.0   # y-center for HR (Multimodal) bar (top)
y_nonhr = 1.0   # y-center for Non-HR (Non-Multimodal) bar (bottom)

function hbar_segment(y, x0, x1, h)
    Shape([x0, x1, x1, x0], [y-h/2, y-h/2, y+h/2, y+h/2])
end

plot2 = plot(
    legend = false,
    xlims  = (0, 100),
    ylims  = (0.3, 2.7),
    yticks = ([y_nonhr, y_hr], ["Non-HR", "HR"]),
    xticks = ([0, 100], ["0%", "100%"]),
    framestyle = :box,
    grid = false,
    size = (500, 400),
    xtickfont  = font(14, "Helvetica"),
    ytickfont  = font(14, "Helvetica"),
    bottom_margin = 5Plots.mm,
    left_margin   = 8Plots.mm,
    right_margin   = 8Plots.mm,
    xmirror = true  
)

# HR (Multimodal) bar: red first, then gray 
plot!(plot2, hbar_segment(y_hr, 0, hr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_hr, hr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

# Non-HR (Non-Multimodal) bar: red first, then gray on top
plot!(plot2, hbar_segment(y_nonhr, 0, nonhr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_nonhr, nonhr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

display(plot2)

# ------------------------------------------------------------
# 9. SAVE Horizontal Stacked Bar Plot
# ------------------------------------------------------------
# savefig(plot2, "figure_2E_style_NegFB.svg")
# savefig(plot2, "figure_2E_style_NegFB.png")

println("Analysis complete. Stacked horizontal bar plot saved.")

In [ ]:
# Fig 2E heatmap, built from the Negative-FB contingency table computed above
h_E = plot_contingency_heatmap(a, b, c, d;
        with_label="With NegFB", without_label="Without NegFB",
        p_val_str = p_two < 0.0001 ? "< 0.0001" : "= " * string(round(p_two, sigdigits=4)),
        odds_ratio = OR)
display(h_E)
# savefig(h_E, "contingency_heatmap_NegFB.png")
# savefig(h_E, "contingency_heatmap_NegFB.svg")


### Shared utilities for Fig 2F–K : 3-category edge-density stratified Fisher analysis + panel plotting

In [ ]:
###############################################################
# Shared utilities used by Fig 2F, 2G, 2H, 2I, 2J, 2K
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using HypothesisTests
using Printf
using Plots
using Measures
using Colors

gr()

Plots.default(
    xtickfont      = font(22, "Helvetica Bold"),
    ytickfont      = font(22, "Helvetica Bold"),
    xguidefont     = font(24, "Helvetica Bold"),
    yguidefont     = font(24, "Helvetica Bold"),
    legendfont     = font(20, "Helvetica Bold"),
    titlefont      = font(24, "Helvetica Bold"),
    tick_direction = :out,
    linewidth      = 2.0,
    framestyle     = :box,
    grid           = false,
    dpi            = 600,
)

# ------------------------------------------------------------
# 3-category edge-density stratification: 1-3 / 4-6 / 7-9 edges
# ------------------------------------------------------------
const EDGE_BINS = [
    (min=1, max=3, label="1-3"),
    (min=4, max=6, label="4-6"),
    (min=7, max=9, label="7-9"),
]

"""
    run_fisher_by_edgebins(df, category_col, present_val, absent_val, analysis_name)

Runs a Fisher's exact test within each edge-density bin in EDGE_BINS. `df`
must contain a `:total_edges` column and an `:is_multimodal` (0/1) column,
plus `category_col` holding `present_val` / `absent_val` (or `true`/`false`
for motif flags such as `:has_ffl`, `:has_cffl`, `:has_iffl`).

Returns (bin_labels, log_or_list, p_two_sided_list, total_network_counts),
one entry per non-empty bin, and prints the full table so the numbers
behind each dot in F-K are auditable.
"""
function run_fisher_by_edgebins(df::DataFrame, category_col::Symbol, present_val, absent_val, analysis_name::String)
    bin_labels, log_or_list, p_list, total_net_counts = String[], Float64[], Float64[], Int[]

    println("\n" * "="^95)
    println("  STATISTICAL ANALYSIS TABLE: $analysis_name")
    println("="^95)
    @printf("%-6s | %-5s | %-5s | %-5s | %-5s | %-7s | %-8s | %-10s | %-10s\n",
            "Bin", "a", "b", "c", "d", "Total", "OR", "Log OR", "p (2-side)")
    println("-"^95)

    for bset in EDGE_BINS
        subset = filter(row -> row.total_edges >= bset.min && row.total_edges <= bset.max, df)
        total_nets = nrow(subset)
        total_nets == 0 && continue

        a     = sum((subset[!, category_col] .== present_val) .& (subset.is_multimodal .== 1))
        b_val = sum((subset[!, category_col] .== present_val) .& (subset.is_multimodal .== 0))
        c     = sum((subset[!, category_col] .== absent_val)  .& (subset.is_multimodal .== 1))
        d     = sum((subset[!, category_col] .== absent_val)  .& (subset.is_multimodal .== 0))

        # Haldane-Anscombe correction so log(OR) is defined even with zero cells
        a_c = a == 0     ? 0.5 : Float64(a)
        b_c = b_val == 0 ? 0.5 : Float64(b_val)
        c_c = c == 0     ? 0.5 : Float64(c)
        d_c = d == 0     ? 0.5 : Float64(d)

        OR    = (a_c * d_c) / (b_c * c_c)
        logOR = log(OR)

        fet = FisherExactTest(Int(a), Int(b_val), Int(c), Int(d))
        p2  = pvalue(fet; tail=:both)

        @printf("%-6s | %-5d | %-5d | %-5d | %-5d | %-7d | %-8.4f | %-10.4f | %-10.4e\n",
                bset.label, a, b_val, c, d, total_nets, OR, logOR, p2)

        push!(bin_labels, bset.label)
        push!(log_or_list, logOR)
        push!(p_list, p2)
        push!(total_net_counts, total_nets)
    end
    println("="^95)

    return bin_labels, log_or_list, p_list, total_net_counts
end

"""
    plot_logor_panel(bin_labels, log_or_list, p_list, total_net_counts, panel_title;
                      bar_color=:gray55, dot_color=:red,
                      count_ylim=(10^1, 10^4), or_ylim=nothing)

Bar chart of network counts (log10 y-axis, left) with a twin-axis scatter of
log-odds-ratios (right): filled dot = p<0.05, open dot = p>=0.05, dashed line
at log(OR)=0. `or_ylim` auto-expands to fit the data when not given. One
function drives every panel in F, G, H, I, J, K so the style always matches
across panels.
"""
function plot_logor_panel(bin_labels, log_or_list, p_list, total_net_counts, panel_title;
                           bar_color=:gray55, dot_color=:red,
                           count_ylim=(10^1, 10^4), or_ylim=nothing)

    n = length(bin_labels)
    x_pos = 1:n
    is_sig = p_list .<= 0.05
    sig_idx, nonsig_idx = findall(is_sig), findall(.!is_sig)

    if or_ylim === nothing
        lo = min(-2.0, minimum(log_or_list) - 0.5)
        hi = max(6.0, maximum(log_or_list) + 0.5)
        or_ylim = (lo, hi)
    end

    p_main = bar(
        x_pos, total_net_counts,
        xlabel = "Number of Edges", ylabel = "Count of Networks",
        yguidefontcolor = :black, ytickfontcolor = :black,
        title = panel_title, color = bar_color, alpha = 0.7,
        xticks = (x_pos, bin_labels), xlim = (0.5, n + 0.5),
        yscale = :log10, ylim = count_ylim,
        fillrange = count_ylim[1],
        label = "", legend = false, framestyle = :box, linewidth = 2,
        foreground_color_border = :black,
        left_margin = 12Plots.mm, right_margin = 18Plots.mm,
        top_margin = 8Plots.mm, bottom_margin = 12Plots.mm,
    )

    p_twin = twinx(p_main)
    plot!(p_twin,
        xlim = (0.5, n + 0.5), ylim = or_ylim,
        ylabel = "Log Odds Ratio", yguidefontcolor = dot_color, ytickfontcolor = dot_color,
        xticks = false, legend = false, framestyle = :box, linewidth = 2,
        foreground_color_border = :black,
    )

    if !isempty(nonsig_idx)
        scatter!(p_twin, x_pos[nonsig_idx], log_or_list[nonsig_idx],
            color=:white, markerstrokecolor=dot_color, markerstrokewidth=2,
            markershape=:circle, markersize=10, label="")
    end
    if !isempty(sig_idx)
        scatter!(p_twin, x_pos[sig_idx], log_or_list[sig_idx],
            color=dot_color, markershape=:circle, markersize=10, label="")
    end
    hline!(p_twin, [0.0], linestyle=:dash, color=:black, linewidth=1.5, label="")

    return p_main
end

println("Shared utilities loaded: EDGE_BINS, run_fisher_by_edgebins(), plot_logor_panel()")


### Fig 2F, 2G, 2H : Edgewise analysis for FB, PosFB, NegFB (3-category edge-density stratification)

In [ ]:
###############################################################
# Fig 2F, 2G, 2H : FB / PosFB / NegFB edgewise analysis
###############################################################

println("Loading data for Fig 2F-H...")

csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1_filtered.csv"
df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

dir = raw"F:\HR_Project\noise1"
jld_path     = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]

println("Data loaded.")

# ------------------------------------------------------------
# Classifiers: any FB (F), positive FB (G), negative FB (H)
# ------------------------------------------------------------
function classify_feedback(mat)
    fb = any(mat[i,i] != 0 for i in 1:3) ||
         any((mat[i,j] != 0 && mat[j,i] != 0) for (i,j) in [(1,2),(1,3),(2,3)]) ||
         (mat[1,2]!=0 && mat[2,3]!=0 && mat[3,1]!=0) ||
         (mat[1,3]!=0 && mat[3,2]!=0 && mat[2,1]!=0)
    total_edges = count(x -> x != 0, mat)
    return fb ? ("FB Present", total_edges) : ("FB Absent", total_edges)
end

function classify_pos_feedback(mat)
    pfb = sum(mat[i,i] > 0 for i in 1:3) +
          sum((mat[i,j]*mat[j,i] > 0) for (i,j) in [(1,2),(1,3),(2,3)]) +
          (mat[1,2]*mat[2,3]*mat[3,1] > 0 ? 1 : 0) +
          (mat[1,3]*mat[3,2]*mat[2,1] > 0 ? 1 : 0)
    total_edges = count(x -> x != 0, mat)
    return pfb > 0 ? ("Pos FB Present", total_edges) : ("Pos FB Absent", total_edges)
end

function classify_neg_feedback(mat)
    nfb = sum(mat[i,i] < 0 for i in 1:3) +
          sum((mat[i,j]*mat[j,i] < 0) for (i,j) in [(1,2),(1,3),(2,3)]) +
          (mat[1,2]*mat[2,3]*mat[3,1] < 0 ? 1 : 0) +
          (mat[1,3]*mat[3,2]*mat[2,1] < 0 ? 1 : 0)
    total_edges = count(x -> x != 0, mat)
    return nfb > 0 ? ("Neg FB Present", total_edges) : ("Neg FB Absent", total_edges)
end

function build_feedback_df(all_matrices, net_counts, classifier)
    rows = []
    for id in keys(all_matrices)
        mat = all_matrices[id].adj_matrix
        !is_valid_network(mat) && continue
        cat, tot_edges  = classifier(mat)
        idx             = findfirst(==(id), net_counts.net_id)
        multimodal_val  = isnothing(idx) ? 0.0 : net_counts[idx, :percentage_multimodal]
        push!(rows, (net_id=id, category=cat, is_multimodal=(multimodal_val > 0 ? 1 : 0), total_edges=tot_edges))
    end
    return DataFrame(rows)
end

fb_df  = build_feedback_df(all_matrices, net_counts, classify_feedback)
pfb_df = build_feedback_df(all_matrices, net_counts, classify_pos_feedback)
nfb_df = build_feedback_df(all_matrices, net_counts, classify_neg_feedback)

bl_fb,  lo_fb,  p_fb,  cnt_fb  = run_fisher_by_edgebins(fb_df,  :category, "FB Present",     "FB Absent",     "Feedback Loop (F) Analysis")
bl_pfb, lo_pfb, p_pfb, cnt_pfb = run_fisher_by_edgebins(pfb_df, :category, "Pos FB Present", "Pos FB Absent", "Positive Feedback (G) Analysis")
bl_nfb, lo_nfb, p_nfb, cnt_nfb = run_fisher_by_edgebins(nfb_df, :category, "Neg FB Present", "Neg FB Absent", "Negative Feedback (H) Analysis")

p_F = plot_logor_panel(bl_fb,  lo_fb,  p_fb,  cnt_fb,  "F: Feedback Loop")
p_G = plot_logor_panel(bl_pfb, lo_pfb, p_pfb, cnt_pfb, "G: Positive Feedback Loop")
p_H = plot_logor_panel(bl_nfb, lo_nfb, p_nfb, cnt_nfb, "H: Negative Feedback Loop")

fig_FGH = plot(p_F, p_G, p_H, layout=(1, 3), size=(3000, 700))
display(fig_FGH)

# savefig(fig_FGH, "fig2_FGH_feedback_edgewise.png")
# savefig(fig_FGH, "fig2_FGH_feedback_edgewise.svg")
println("Done: Fig 2F-H.")

### Fig 2I : Contingency table & edgewise log-odds for FFL (any: coherent + incoherent)

In [ ]:
###############################################################
# Fig 2I : FFL (any) contingency table + edgewise log-odds panel
###############################################################

println("Loading data for Fig 2I...")
# Reloaded defensively so this cell also works if run standalone.

csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1_filtered.csv"
df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

dir = raw"F:\HR_Project\noise1"
network_data = load(joinpath(dir, "unique_networks_table.jld2"))
all_matrices = network_data["network_table"]

# ------------------------------------------------------------
# Motif detectors (coherent / incoherent / any FFL)
# ------------------------------------------------------------
function is_coherent_ffl(mat)
    nodes = [1, 2, 3]
    for i in nodes, j in nodes, k in nodes
        (j == i || k == i || k == j) && continue
        if mat[j, i] != 0 && mat[k, j] != 0 && mat[k, i] != 0
            sign(mat[k, i]) == sign(mat[j, i] * mat[k, j]) && return true
        end
    end
    return false
end

function is_incoherent_ffl(mat)
    nodes = [1, 2, 3]
    for i in nodes, j in nodes, k in nodes
        (j == i || k == i || k == j) && continue
        if mat[j, i] != 0 && mat[k, j] != 0 && mat[k, i] != 0
            sign(mat[k, i]) != sign(mat[j, i] * mat[k, j]) && return true
        end
    end
    return false
end

is_any_ffl(mat) = is_coherent_ffl(mat) || is_incoherent_ffl(mat)

# ------------------------------------------------------------
# Build DataFrame (used for both the contingency counts and the
# edgewise Fisher test below, and re-used by the Fig 2J/K cell)
# ------------------------------------------------------------
function build_motif_df(all_matrices, net_counts)
    rows = []
    for id in keys(all_matrices)
        mat = all_matrices[id].adj_matrix
        !is_valid_network(mat) && continue
        idx = findfirst(==(id), net_counts.net_id)
        cnt = isnothing(idx) ? 0 : net_counts[idx, :count]
        push!(rows, (net_id=id, total_edges=count(x -> x != 0, mat),
                      has_ffl=is_any_ffl(mat), has_cffl=is_coherent_ffl(mat),
                      has_iffl=is_incoherent_ffl(mat), is_multimodal=(cnt > 0 ? 1 : 0)))
    end
    return DataFrame(rows)
end

motif_df = build_motif_df(all_matrices, net_counts)

# ------------------------------------------------------------
# Contingency table: HR (is_multimodal) x FFL present/absent
# ------------------------------------------------------------
n_hr_ffl      = sum((motif_df.is_multimodal .== 1) .& motif_df.has_ffl)
n_hr_noffl    = sum((motif_df.is_multimodal .== 1) .& .!motif_df.has_ffl)
n_nonhr_ffl   = sum((motif_df.is_multimodal .== 0) .& motif_df.has_ffl)
n_nonhr_noffl = sum((motif_df.is_multimodal .== 0) .& .!motif_df.has_ffl)

fet_ffl = FisherExactTest(n_hr_ffl, n_hr_noffl, n_nonhr_ffl, n_nonhr_noffl)
p_ffl   = pvalue(fet_ffl; tail=:both)
println("FFL contingency: HR/FFL=$n_hr_ffl HR/noFFL=$n_hr_noffl NonHR/FFL=$n_nonhr_ffl NonHR/noFFL=$n_nonhr_noffl  p=$p_ffl")

counts_I     = [n_hr_ffl  n_hr_noffl; n_nonhr_ffl n_nonhr_noffl]
log_counts_I = log10.(counts_I .+ 1)

h_I = heatmap(
    [1, 2], [1, 2], log_counts_I,
    c = :YlGnBu, colorbar = true, colorbar_pad = 15,
    framestyle = :box, grid = false, linewidth = 2.0, aspect_ratio = 1, size = (750, 600),
    xlims = (0.5, 2.5), ylims = (0.5, 2.5),
    xticks = ([1, 2], ["With FFL", "Without FFL"]),
    yticks = ([1, 2], ["HR-Enabling \nNetworks", "HR-Disabling \nNetworks"]),
    yflip = true,
    xtickfont = font(14, "Helvetica Bold"), ytickfont = font(14, "Helvetica Bold"),
    xguidefont = font(18, "Helvetica Bold"), yguidefont = font(18, "Helvetica Bold"),
    left_margin = 25Plots.mm, right_margin = 20Plots.mm, top_margin = 15Plots.mm, bottom_margin = 15Plots.mm,
    colorbar_title = "\nlog10(count + 1)"
)
annotate!(1, 1, text("n = $n_hr_ffl",      12, n_hr_ffl      > 400 ? :white : :black, :bold))
annotate!(2, 1, text("n = $n_hr_noffl",    12, n_hr_noffl    > 400 ? :white : :black, :bold))
annotate!(1, 2, text("n = $n_nonhr_ffl",   12, n_nonhr_ffl   > 400 ? :white : :black, :bold))
annotate!(2, 2, text("n = $n_nonhr_noffl", 12, n_nonhr_noffl > 400 ? :white : :black, :bold))

# ------------------------------------------------------------
# Edgewise log-odds panel (density-stratified, 1-3 / 4-6 / 7-9)
# ------------------------------------------------------------
bl_ffl, lo_ffl, p_ffl_list, cnt_ffl = run_fisher_by_edgebins(motif_df, :has_ffl, true, false, "FFL (I) Analysis")
p_I_chart = plot_logor_panel(bl_ffl, lo_ffl, p_ffl_list, cnt_ffl, "I: Feed-Forward Loop"; count_ylim=(10^0, 10^4))

fig_I = plot(h_I, p_I_chart, layout = (1, 2), size = (1700, 650))
display(fig_I)

# savefig(fig_I, "fig2_I_ffl.png")
# savefig(fig_I, "fig2_I_ffl.svg")
println("Done: Fig 2I.")

### Fig 2J, 2K : Edgewise analysis for coherent FFL (cFFL) and incoherent FFL (iFFL)

In [ ]:
###############################################################
# Fig 2J, 2K : cFFL and iFFL edgewise analysis
# (3-category edge-density stratification: 1-3 / 4-6 / 7-9)
# Reuses `motif_df` (and is_coherent_ffl / is_incoherent_ffl) built in the
# Fig 2I cell above -- run that cell first.
###############################################################

if !@isdefined(motif_df)
    error("Run the Fig 2I cell first (it builds `motif_df`).")
end

bl_c, lo_c, p_c, cnt_c = run_fisher_by_edgebins(motif_df, :has_cffl, true, false, "cFFL (J) Analysis")
bl_i, lo_i, p_i, cnt_i = run_fisher_by_edgebins(motif_df, :has_iffl, true, false, "iFFL (K) Analysis")

p_J = plot_logor_panel(bl_c, lo_c, p_c, cnt_c, "J: Coherent FFL")
p_K = plot_logor_panel(bl_i, lo_i, p_i, cnt_i, "K: Incoherent FFL")

fig_JK = plot(p_J, p_K, layout = (1, 2), size = (2000, 700))
display(fig_JK)

# savefig(fig_JK, "fig2_JK_cffl_iffl.png")
# savefig(fig_JK, "fig2_JK_cffl_iffl.svg")
println("Done: Fig 2J-K.")
